# Test the opt-in load selector

Compare historical workload scenarios with compatible service throughput. Use the sourced
March 2025 dialysis scenario, a supplied CSV, or the built-in fictional example. These are
planning comparisons using straight-line distance.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# Fill this in if the notebook's working directory is outside the repository.
REPO_ROOT = os.environ.get("DATATHON_REPO_ROOT") or None
cwd = Path.cwd().resolve()
ROOT = Path(REPO_ROOT).expanduser().resolve() if REPO_ROOT else next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "non-ml" / "config" / "project.json").is_file()),
    None,
)
if ROOT is None or not (ROOT / "non-ml" / "config" / "project.json").is_file():
    raise RuntimeError("Set REPO_ROOT to the directory that contains non-ml/config/project.json.")
sys.path.insert(0, str(ROOT / "non-ml" / "src"))

from facility_tool.workspace import Store, read_config, show

config = read_config(ROOT)
MODE = os.environ.get("DATATHON_MODE", config["input_mode"])
STORE = Store(ROOT, MODE, config["as_of"])
print(f"Input mode: {MODE} | Evidence date: {config['as_of']}")

## Run matched comparisons
Keep the service, facility evidence, and location fixed. The opt-in selector considers
only adequately supported options of comparable scope within the extra travel limit.
Its final score can trade extra travel against a condition-specific demand/capacity
ratio. Compare only compatible unit, period, and source basis. Scenario or historical
pressure is labeled as such. Total beds alone won't stand in for dialysis slots.
The dialysis input uses observed March 2025 haemodialysis sessions and predicted monthly
throughput. Each predicted capacity is flagged and linked to its source and method.
See [the implementation report](../LOAD_BALANCING.md) for the source and evaluation.

In [ ]:
import json

from facility_tool.lookup import lookup, resolve_service
from facility_tool.recommend import simulate_spread

CONCERN = config['lookup']['service']
LOCATION = config['lookup']['location']
SCENARIO_CSV = None  # Optional table of comparable demand and capacity.
USE_DIALYSIS_EVIDENCE = MODE != 'demo' and resolve_service(CONCERN, config['service_aliases']) == 'dialysis'
REQUESTS = 100  # Fictional suggestions in the optional replay. No patient records.
if SCENARIO_CSV:
    pressure = pd.read_csv(SCENARIO_CSV)
elif USE_DIALYSIS_EVIDENCE:
    evidence = json.loads((ROOT / 'non-ml/caseload/load_evidence.json').read_text())
    pressure = pd.DataFrame(evidence['pressure_inputs'])
    show(pressure[['facility_id', 'demand', 'capacity', 'capacity_predicted', 'period']])
    replay = json.loads((ROOT / 'non-ml/caseload/load_planning_replay.json').read_text())
    show(pd.DataFrame([replay['summary']]))
else:
    pressure = STORE.read('pressure_scenarios', optional=True)
common = dict(concern=CONCERN, location=LOCATION, traveltime_enabled=False,
              pressure_inputs=pressure, load_context='planning')
baseline = lookup(STORE, config, **common, spread_demand=False)
opt_in = lookup(STORE, config, **common, spread_demand=True)
columns = ['mode', 'name', 'rank', 'distance_km', 'extra_km_vs_baseline', 'pressure_ratio',
           'pressure_capacity_predicted', 'pressure_capacity_method', 'pressure_capacity_source_url',
           'pressure_grade', 'travel_penalty', 'pressure_penalty', 'weighted_score', 'weighted_status']
show(baseline['summary'])
show(opt_in['summary'])
show(opt_in['rows'].loc[opt_in['rows']['kind'].eq('recommendation'), columns], limit=25)

## Sensitivity and a separate capacity-weighted replay
Compare first choices across multiple travel weights. The replay demonstrates how
a fixed batch of hypothetical suggestions might be spread across sites. Its output
measures recommendations shown, with no claim about visits or actual crowding.

In [ ]:
if pressure.empty:
    print('No comparable demand/capacity inputs. The ordinary ranking is available.')
else:
    sensitivity = []
    for weight in (0.25, 0.5, 0.75):
        scenario = lookup(STORE, config, **common, spread_demand=True, travel_weight=weight)
        first = scenario['rows'].loc[scenario['rows']['kind'].eq('recommendation') &
                                     scenario['rows']['rank'].eq(1)]
        sensitivity.append(first[['mode', 'name', 'extra_km_vs_baseline', 'pressure_ratio',
                                  'weighted_score', 'weighted_status']].assign(travel_weight=weight))
    show(pd.concat(sensitivity, ignore_index=True))
    if MODE == 'demo' or SCENARIO_CSV:
        base_candidates = baseline['trace'].loc[baseline['trace']['mode'].eq('confirmed')]
        show(simulate_spread(base_candidates, pressure, baseline['service'], requests=REQUESTS,
                             max_extra_km=config['lookup']['max_extra_km']))

## Historical replay across Care needed options

The checked replay aligns published reports and backtested caseload predictions by
branch, activity definition and period. Its coverage distinguishes origins screened
from cases with enough evidence to compare rankings. See the [replay report](../caseload/CARE_OPTIONS_REPLAY.md)
for prediction flags, inputs and reproduction commands.

In [ ]:
care_replay = json.loads((ROOT / 'non-ml/caseload/care_options_replay.json').read_text(encoding='utf-8'))
show(pd.DataFrame([care_replay['summary']]))
show(pd.DataFrame(care_replay['coverage']), limit=25)
changes = [dict(care_needed=group['care_needed'], period=group['period_start'],
                origin=case['origin']['name'], **move)
           for group in care_replay['cohorts'] for case in group['cases']
           for move in (case['rank_changes'] or [])]
if changes:
    show(pd.DataFrame(changes))
else:
    print('No supported ranking changes in the expanded historical replay.')

## Network-wide caseload scenarios

All release rows have numerical scenarios. A conditional value assumes a branch could
provide the specified service and does not verify that it does. This notebook compares
only branches with both independent lookup eligibility and exact historical scope evidence.
The [network report](../caseload/NETWORK_CASELOAD.md) gives a separate explicit hypothetical mode.

In [ ]:
from facility_tool.network_load import read_network_pressure, replay_network_load

NETWORK_FORECASTS = ROOT / 'non-ml/caseload/network_caseload.csv.gz'
RUN_NETWORK_PLANNING = False
if RUN_NETWORK_PLANNING:
    options = dict(concern=CONCERN, location=LOCATION, facility_sort='closest',
                   traveltime_enabled=False, load_context='planning', network_forecasts=NETWORK_FORECASTS)
    network_ranked = lookup(STORE, config, **options, spread_demand=True)
    show(network_ranked['summary'])
    load_status = network_ranked['summary'].get('load_status', pd.Series(dtype=str))
    if load_status.isin(['non_emergency_only', 'no_comparable_load_modality']).any():
        print('The request does not qualify for this planning scenario.')
    else:
        network_trace = network_ranked['trace'].query("mode == 'confirmed' and decision == 'eligible'")
        network_inputs = read_network_pressure(NETWORK_FORECASTS, network_ranked['service'],
                                               network_trace['facility_id'])
        network_inputs = network_inputs.loc[network_inputs['service_evidence'].eq('reported_activity_in_scope')]
        supported = network_trace.loc[network_trace['facility_id'].isin(network_inputs['facility_id'])]
        show(replay_network_load(supported, network_inputs, requests=REQUESTS), limit=25)